## **Downloading**

In [ ]:
!pip install biopython pandas networkx matplotlib tqdm py4cytoscape # python-louvain

In [3]:
!wget -O - https://ftp.ebi.ac.uk/pub/databases/metagenomics/peptide_database/current_release/mgy_proteins_1.fa.gz | gunzip | head -n 20000 > subset.fasta

--2026-03-12 13:15:16--  https://ftp.ebi.ac.uk/pub/databases/metagenomics/peptide_database/current_release/mgy_proteins_1.fa.gz
Resolving ftp.ebi.ac.uk (ftp.ebi.ac.uk)... 193.62.193.165
Connecting to ftp.ebi.ac.uk (ftp.ebi.ac.uk)|193.62.193.165|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 11412619458 (11G) [application/x-gzip]
Saving to: ‘STDOUT’

-                     0%[                    ]   1.13M  2.46MB/s               
gzip: stdout: Broken pipe
-                     0%[                    ]   1.56M  3.34MB/s    in 0.5s    


Cannot write to ‘-’ (Success).


In [6]:
!grep -c ">" subset.fasta

10000


In [8]:
!head subset.fasta

>MGYP004668706029 FL=1
MNIVRDLAVILISAGVFTIISKALKQPLILGYILAGFLIGPNVTFFPGITNEATMHQWSEIGIIFLMFGLGLEFSFKKLLKVGGSAIVTAAVKFVGVFIIGFVTAQALSWSFMESVFLGGLLSMSSTMVVLKSYDDLGLKNKPYAGVVFGTLVVEDLIAILLMVLLSTMAVSQSFAGKELIMNIAKLVFFLILWFLVGIYVIPTLLKKAKEYLNDEILLIVSIGLCFGMVALATSVGFSSALGAFVMGSILAETSESEHIDHIVEPIKNLFGAIFFVSVGMMVAPSVIAEHWAMILLLSVIVIISHIIFAGAGIILTGKGLDNAVHTGFSLAQLGEFGFIIAGVGCTLGVMRDFIYPVIIAVSVITTFTTPFMIKLADPCYNLLNKRLPAKWIDRLAQMNDSGKQTAAEHNEWKTLLNAYFTRIVLYGVILIAIYIGSKLYLRPAVEKFSPELSVTIHKVIEVGITLAAMLPFLFGLGVHSGSISKSAPKLLKEKPSNIWPLMGLIFARSFLAVGIVLAVLSSYFHLAGWTVLAIFFAGIIFILFARRSIHKYSALEMRFLSNFNEKEENERRSKPVASSVSQKLADYDVHTETLTISQNSTYAGKALKDIPFRAETGANIIKITRGSMNMIVPSGDVSLFPGDQMLAVGTSAQLESLRNMMACAVAPEIQDSGNSFKIVPEALTEESFLTGKTLRGTNLRKYHCMVISVLRGSEFITNPEPDFRFQAGDTVWIAGNLAELETV
>MGYP000862152061 FL=0
LNGDQTNRERLQVLVDESATKDSNAKYYNADAEIQAVYDKAVEEAKATLAKENVTQAEVDAAKAKLQEAKDALNGVDTNKEALQNLADESNAKDSNAKYYNADADKQEAYNKAVEEAKAVLSKENVTQAEIDAAKAKLQEAKDALNGADTNKTELEKQVELKNSDEVTKDSKYYNASEESLLNYYNVVKKAEETLAKPNVTQDEVDKMV

In [10]:
!grep ">" subset.fasta | sed 's/>//' > protein_ids.txt

In [12]:
!wget -qO- https://ftp.ebi.ac.uk/pub/databases/metagenomics/peptide_database/current_release/mgy_biomes.tsv.gz \
| zcat | grep -F -f protein_ids.txt > metadata.tsv

In [13]:
!ls -lh metadata.tsv

-rw-r--r-- 1 root root 5.7K Mar 12 13:30 metadata.tsv


In [14]:
!head metadata.tsv

MGYP000225593703	2	root:Host-associated:Human:Digestive system
MGYP000225593703	5	root:Host-associated:Human:Digestive system:Hindgut
MGYP000225593703	12	root:Host-associated:Human:Digestive system:Large intestine
MGYP000225593703	68	root:Host-associated:Human:Digestive system:Large intestine:Fecal
MGYP000225593703	1	root:Host-associated:Mammals:Digestive system:Fecal
MGYP000225593703	9	root:Host-associated:Mammals:Digestive system:Large intestine:Fecal
MGYP000229914253	1	root:Host-associated:Human:Digestive system:Large intestine
MGYP000229914253	1	root:Mixed
MGYP000266133040	4	root:Host-associated:Human:Digestive system:Large intestine:Fecal
MGYP000266133040	1	root:Host-associated:Mammals:Digestive system:Large intestine:Fecal


In [15]:
!wc -l metadata.tsv

81 metadata.tsv


## **Processing**

In [3]:
from Bio import SeqIO

sequences = []

for record in SeqIO.parse("/kaggle/input/datasets/sarthakkalia45/small-subset-mgnify/subset.fasta", "fasta"):
    
    sequences.append({
        "protein_id": record.id,
        "sequence": str(record.seq)
    })

print("Total proteins:", len(sequences))

Total proteins: 10000


In [4]:
import pandas as pd

metadata = pd.read_csv("/kaggle/input/datasets/sarthakkalia45/small-subset-mgnify/metadata.tsv", sep="\t", header=None)
metadata.columns = ["protein_id", "count", "biome"]
print(metadata.head())

         protein_id  count                                              biome
0  MGYP000225593703      2        root:Host-associated:Human:Digestive system
1  MGYP000225593703      5  root:Host-associated:Human:Digestive system:Hi...
2  MGYP000225593703     12  root:Host-associated:Human:Digestive system:La...
3  MGYP000225593703     68  root:Host-associated:Human:Digestive system:La...
4  MGYP000225593703      1  root:Host-associated:Mammals:Digestive system:...


In [5]:
metadata_map = dict(
    zip(metadata.protein_id, metadata.biome)
)

for seq in sequences:
    seq["biome"] = metadata_map.get(seq["protein_id"], "unknown")

In [6]:
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

records = []

for seq in sequences:
    
    record = SeqRecord(
        Seq(seq["sequence"]),
        id=seq["protein_id"],
        description=""
    )

    records.append(record)

SeqIO.write(records, "proteins.fasta", "fasta")

10000

In [ ]:
!wget -qO- https://micromamba.snakepit.net/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
!./bin/micromamba install -y -n base -c conda-forge -c bioconda mmseqs2

In [ ]:
!apt-get update
!apt-get install -y mmseqs2

In [ ]:
!mmseqs --help

In [11]:
import subprocess
import os
output_dir="/kaggle/working/output"
def run_mmseqs(input_fasta, output_dir):

    os.makedirs(output_dir, exist_ok=True)

    db = f"{output_dir}/proteinDB"
    result = f"{output_dir}/resultDB"
    tmp = f"{output_dir}/tmp"
    tsv = f"{output_dir}/similarity.tsv"

    print("Creating MMseqs database...")
    subprocess.run(["mmseqs", "createdb", input_fasta, db], check=True)

    print("Running similarity search...")
    subprocess.run([
        "mmseqs",
        "search",
        db,
        db,
        result,
        tmp,
        "--min-seq-id",
        "0.3",
        "-c",
        "0.8"
    ], check=True)

    print("Converting results to TSV...")
    subprocess.run([
        "mmseqs",
        "convertalis",
        db,
        db,
        result,
        tsv
    ], check=True)

    print("Done! Output file:", tsv)

In [12]:
run_mmseqs("proteins.fasta", "mmseqs_output")

Creating MMseqs database...
createdb proteins.fasta mmseqs_output/proteinDB 

MMseqs Version:       	13-45111+ds-2
Database type         	0
Shuffle input database	true
Createdb mode         	0
Write lookup file     	1
Offset of numeric ids 	0
Compressed            	0
Verbosity             	3

Converting sequences
[
Time for merging to proteinDB_h: 0h 0m 0s 3ms
Time for merging to proteinDB: 0h 0m 0s 6ms
Database type: Aminoacid
Time for processing: 0h 0m 0s 42ms
Running similarity search...
Create directory mmseqs_output/tmp
search mmseqs_output/proteinDB mmseqs_output/proteinDB mmseqs_output/resultDB mmseqs_output/tmp --min-seq-id 0.3 -c 0.8 

MMseqs Version:                        	13-45111+ds-2
Substitution matrix                    	nucl:nucleotide.out,aa:blosum62.out
Add backtrace                          	false
Alignment mode                         	2
Alignment mode                         	0
Allow wrapped scoring                  	false
E-value threshold                      	0

In [13]:
import pandas as pd
import networkx as nx

df = pd.read_csv("mmseqs_output/similarity.tsv", sep="\t", header=None)

df = df[[0,1,2]]
df.columns = ["query","target","identity"]

G = nx.Graph()

for _,row in df.iterrows():
    
    if row.identity >= 0.3:
        G.add_edge(row.query,row.target,weight=row.identity)

print("Nodes:",G.number_of_nodes())
print("Edges:",G.number_of_edges())

Nodes: 10000
Edges: 25245


In [14]:
for node in G.nodes():
    biome = metadata_map.get(node, "unknown")
    G.nodes[node]["biome"] = biome

In [15]:
print(G.nodes["MGYP000229914253"])

{'biome': 'root:Mixed'}


In [16]:
threshold = 0.4

edges_to_remove = [
    (u,v) for u,v,d in G.edges(data=True)
    if d["weight"] < threshold
]

G.remove_edges_from(edges_to_remove)

print("Nodes:",G.number_of_nodes())
print("Edges:",G.number_of_edges())

Nodes: 10000
Edges: 18648


In [7]:
!pip install python-louvain --no-cache-dir

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 7.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for python-louvain: filename=python_louvain-0.16-py3-none-any.whl size=9388 sha256=690f27e58170addc3b74e838e1e7da0a72297fd4966b2198d25cd9d77da72b3d
  Stored in directory: /tmp/pip-ephem-wheel-cache-34h5efbq/wheels/40/f1/e3/485b698c520fa0baee1d07897abc7b8d6479b7d199ce96f4af
Successfully built python-louvain


In [17]:
import community as community_louvain

partition = community_louvain.best_partition(G)

In [18]:
for node,cluster in partition.items():
    G.nodes[node]["cluster"] = cluster

In [20]:
print("Communities:", len(set(partition.values())))

Communities: 6728


In [21]:
import pandas as pd

cluster_df = pd.DataFrame(list(partition.items()),
                          columns=["protein","cluster"])

print(cluster_df.cluster.value_counts().head())

cluster
1989    69
190     67
27      52
70      51
196     46
Name: count, dtype: int64


In [30]:
cluster_df.to_csv("cluster_df.csv", index=False)

In [58]:
nodes = []

for n,d in G.nodes(data=True):
    
    nodes.append({
        "protein":n,
        "biome":d["biome"],
        "cluster":d["cluster"]
    })

pd.DataFrame(nodes).to_csv("nodes.csv",index=False)

In [59]:
edges = []

for u,v,d in G.edges(data=True):
    
    edges.append({
        "source":u,
        "target":v,
        "weight":d["weight"]
    })

pd.DataFrame(edges).to_csv("edges.csv",index=False)

### **Embedding-based similarity graph**

In [ ]:
!pip install torch transformers biopython scikit-learn tqdm

In [22]:
from Bio import SeqIO

def load_sequences(fasta_file, max_len=1022):
    sequences = []
    
    for record in SeqIO.parse(fasta_file, "fasta"):
        seq = str(record.seq)[:]
        
        sequences.append({
            "id": record.id,
            "sequence": seq
        })
    
    return sequences


sequences = load_sequences("/kaggle/input/datasets/sarthakkalia45/small-subset-mgnify/subset.fasta")
print("Loaded:", len(sequences))

Loaded: 10000


In [23]:
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "facebook/esm2_t6_8M_UR50D"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/31.4M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
pooler.dense.weight         | MISSING    | 
pooler.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


EsmModel(
  (embeddings): EsmEmbeddings(
    (word_embeddings): Embedding(33, 320, padding_idx=1)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): EsmEncoder(
    (layer): ModuleList(
      (0-5): 6 x EsmLayer(
        (attention): EsmAttention(
          (self): EsmSelfAttention(
            (query): Linear(in_features=320, out_features=320, bias=True)
            (key): Linear(in_features=320, out_features=320, bias=True)
            (value): Linear(in_features=320, out_features=320, bias=True)
            (rotary_embeddings): RotaryEmbedding()
          )
          (output): EsmSelfOutput(
            (dense): Linear(in_features=320, out_features=320, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (LayerNorm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
        )
        (intermediate): EsmIntermediate(
          (dense): Linear(in_features=320, out_features=1280, bias=True)
        )
        (output): EsmOutput(
        

In [24]:
def generate_embeddings(sequences, batch_size=8):
    
    embeddings = {}
    
    for i in tqdm(range(0, len(sequences), batch_size)):
        
        batch = sequences[i:i+batch_size]
        seqs = [item["sequence"] for item in batch]
        ids = [item["id"] for item in batch]
        
        inputs = tokenizer(seqs, return_tensors="pt", padding=True, truncation=True).to(device)
        
        with torch.no_grad():
            outputs = model(**inputs)
        
        # mean pooling over sequence tokens
        batch_embeddings = outputs.last_hidden_state.mean(dim=1).cpu()
        
        for j, protein_id in enumerate(ids):
            embeddings[protein_id] = batch_embeddings[j].numpy()
    
    return embeddings


embeddings = generate_embeddings(sequences)

100%|██████████| 1250/1250 [00:50<00:00, 24.83it/s]


In [25]:
import numpy as np

protein_ids = list(embeddings.keys())
embedding_matrix = np.array([embeddings[p] for p in protein_ids])

print("Embedding shape:", embedding_matrix.shape)

Embedding shape: (10000, 320)


In [26]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(embedding_matrix)
similarity_matrix.shape

(10000, 10000)

In [64]:
import networkx as nx

def build_embedding_graph(protein_ids, sim_matrix, threshold=0.7):
    
    G = nx.Graph()
    
    for i in range(len(protein_ids)):
        G.add_node(protein_ids[i])
    
    for i in range(len(protein_ids)):
        for j in range(i+1, len(protein_ids)):
            
            score = sim_matrix[i][j]
            
            if score >= threshold:
                G.add_edge(protein_ids[i], protein_ids[j], weight=score)
    
    return G


G_embed = build_embedding_graph(protein_ids, similarity_matrix, threshold=0.8)

print("Nodes:", G_embed.number_of_nodes())
print("Edges:", G_embed.number_of_edges())

Nodes: 10000
Edges: 29590473


In [9]:
# thr= 0.7
# # Nodes: 10000
# # Edges: 41554835

In [65]:
import pandas as pd

edges = []

for u, v, d in G_embed.edges(data=True):
    edges.append({
        "source": u,
        "target": v,
        "weight": d["weight"]
    })

pd.DataFrame(edges).to_csv("embedding_edges.csv", index=False)

In [67]:
import community.community_louvain as community_louvain

partition = community_louvain.best_partition(G)

In [68]:
for node,cluster in partition.items():
    G.nodes[node]["cluster"] = cluster

In [69]:
print("Communities:", len(set(partition.values())))

Communities: 6728


In [70]:
import pandas as pd

cluster_df = pd.DataFrame(list(partition.items()),
                          columns=["protein","cluster"])

print(cluster_df.cluster.value_counts().head())

cluster
11     69
191    67
28     52
71     51
197    46
Name: count, dtype: int64


In [71]:
metadata = pd.read_csv("/kaggle/input/datasets/sarthakkalia45/small-subset-mgnify/metadata.tsv", sep="\t", header=None)
metadata.columns = ["protein_id", "count", "biome"]

# create lookup map
biome_map = dict(zip(metadata.protein_id, metadata.biome))

In [72]:
for node in G.nodes():
    G.nodes[node]["biome"] = biome_map.get(node, "unknown")

In [73]:
nodes = []

for n, d in G.nodes(data=True):
    
    nodes.append({
        "id": n,
        "biome": d.get("biome", "unknown"),
        "cluster": d.get("cluster", -1)
    })

nodes_df = pd.DataFrame(nodes)
nodes_df.to_csv("embedding_nodes.csv", index=False)

print("Saved: embedding_nodes.csv")

Saved: embedding_nodes.csv


### **Hybrid approach**

In [54]:
# def build_hybrid_graph(df_mmseqs, protein_ids, sim_matrix, alpha=0.5, threshold=0.5):
    
#     G = nx.Graph()
    
#     # Add nodes
#     for p in protein_ids:
#         G.add_node(p)
    
#     # Add MMseqs edges
#     for _, row in df_mmseqs.iterrows():
#         u = row["query"]
#         v = row["target"]
#         align_score = row["identity"]
        
#         if u in protein_ids and v in protein_ids:
            
#             i = protein_ids.index(u)
#             j = protein_ids.index(v)
            
#             embed_score = sim_matrix[i][j]
            
#             hybrid_score = alpha * align_score + (1 - alpha) * embed_score
            
#             if hybrid_score >= threshold:
#                 G.add_edge(u, v, weight=hybrid_score)
    
#     return G

import networkx as nx

def build_hybrid_graph(df_mmseqs, protein_ids, sim_matrix, alpha=0.5, threshold=0.5):
    
    G = nx.Graph()
    
    # Create fast lookup
    id_to_index = {pid: i for i, pid in enumerate(protein_ids)}
    
    # Add nodes
    for p in protein_ids:
        G.add_node(p)
    
    # Add hybrid edges
    for _, row in df_mmseqs.iterrows():
        
        u = row["query"]
        v = row["target"]
        align_score = row["identity"]
        
        if u in id_to_index and v in id_to_index:
            
            i = id_to_index[u]
            j = id_to_index[v]
            
            embed_score = sim_matrix[i][j]
            
            hybrid_score = alpha * align_score + (1 - alpha) * embed_score
            
            if hybrid_score >= threshold:
                G.add_edge(u, v, weight=hybrid_score)
    
    return G

In [49]:
df_mmseqs = pd.read_csv("mmseqs_output/similarity.tsv", sep="\t", header=None)
df_mmseqs = df_mmseqs[[0, 1, 2]]
df_mmseqs.columns = ["query", "target", "identity"]

print(df_mmseqs.head())

              query            target  identity
0  MGYP004668706029  MGYP004668706029     1.000
1  MGYP003125700799  MGYP003125700799     1.000
2  MGYP003125700799  MGYP003505740598     0.822
3  MGYP003125700799  MGYP006058695731     0.614
4  MGYP003125700799  MGYP001767359898     0.644


In [52]:
G_hybrid = build_hybrid_graph(
    df_mmseqs,
    protein_ids,
    similarity_matrix,
    alpha=0.5,
    threshold=0.5
)

print("Nodes:", G_hybrid.number_of_nodes())
print("Edges:", G_hybrid.number_of_edges())

Nodes: 10000
Edges: 25205


In [56]:
import pandas as pd

edges = []

for u, v, d in G_hybrid.edges(data=True):
    edges.append({
        "source": u,
        "target": v,
        "weight": d["weight"]
    })

edges_df = pd.DataFrame(edges)
edges_df.to_csv("hybrid_edges.csv", index=False)

print("Saved: hybrid_edges.csv")

Saved: hybrid_edges.csv


In [59]:
metadata = pd.read_csv("/kaggle/input/datasets/sarthakkalia45/small-subset-mgnify/metadata.tsv", sep="\t", header=None)
metadata.columns = ["protein_id", "count", "biome"]

# create lookup map
biome_map = dict(zip(metadata.protein_id, metadata.biome))

In [60]:
for node in G_hybrid.nodes():
    G_hybrid.nodes[node]["biome"] = biome_map.get(node, "unknown")

In [61]:
import community as community_louvain
partition = community_louvain.best_partition(G_hybrid)

In [62]:
for node, cluster_id in partition.items():
    G_hybrid.nodes[node]["cluster"] = cluster_id

In [66]:
print("Communities:", len(set(partition.values())))

Communities: 5595


In [63]:
nodes = []

for n, d in G_hybrid.nodes(data=True):
    
    nodes.append({
        "id": n,
        "biome": d.get("biome", "unknown"),
        "cluster": d.get("cluster", -1)
    })

nodes_df = pd.DataFrame(nodes)
nodes_df.to_csv("hybrid_nodes.csv", index=False)

print("Saved: hybrid_nodes.csv")

Saved: hybrid_nodes.csv
